In [19]:
import pandas as pd
import numpy as np
import os

os.makedirs("data", exist_ok=True)

N = 8760  # 1 year hourly
timestamps = pd.date_range(start="2025-01-01", periods=N, freq="h")

rows = []
prev_load = 2

for t in timestamps:

    hour = t.hour
    day = t.dayofweek
    month = t.month

    # temperature simulation
    temperature = np.random.uniform(25, 40)
    humidity = np.random.uniform(40, 90)

    # -----------------------------
    # LOAD PATTERN (REALISTIC)
    # -----------------------------
    # base load
    base_load = 1.5

    # hour effect
    if 6 <= hour <= 9:
        base_load += 1.5
    elif 18 <= hour <= 22:
        base_load += 2.5
    elif 0 <= hour <= 5:
        base_load += 0.5

    # 🔥 temperature effect (AC usage)
    temp_effect = (temperature - 25) * 0.15

    # 🔥 humidity effect
    humidity_effect = (humidity - 50) * 0.02

    # 🔥 weekend effect
    day_effect = 0.5 if day >= 5 else 0

    # 🔥 seasonal effect
    month_effect = 1.0 if month in [4,5,6] else 0

    # combine
    load = base_load + temp_effect + humidity_effect + day_effect + month_effect

    # noise
    load += np.random.normal(0, 0.2)

    load = max(0.5, load)

    rows.append([
        t,
        hour,
        day,
        month,
        temperature,
        humidity,
        prev_load,
        round(load, 2)
    ])

    prev_load = load

df = pd.DataFrame(rows, columns=[
    "timestamp",
    "hour",
    "day",
    "month",
    "temperature",
    "humidity",
    "historical_load",
    "load"
])

df.to_csv("data/load.csv", index=False)

print("⚡ Load dataset created!")

⚡ Load dataset created!


Training

In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

df = pd.read_csv("data/load.csv")

features = [
    "hour",
    "day",
    "month",
    "temperature",
    "humidity",
    "historical_load"
]

X = df[features]
y = df["load"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=200, max_depth=12)
model.fit(X_train, y_train)

# evaluate
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
max_y = max(y_test)
acc = (1 - mae / max_y) * 100

print("\n⚡ LOAD MODEL PERFORMANCE")
print("MAE:", round(mae, 2))
print("R2 :", round(r2, 3))
print("acc:", acc)

joblib.dump(model, "models/load.pkl")

print("\n✅ Load model trained!")


⚡ LOAD MODEL PERFORMANCE
MAE: 0.19
R2 : 0.965
acc: 97.73630840588264

✅ Load model trained!


In [2]:
print("Max y_test:", max(y_test))

Max y_test: 8.38


Predict

In [26]:
import joblib
import pandas as pd

model = joblib.load("models/load.pkl")
data = pd.read_csv('data/load.csv')

# -----------------------------
# CONFIG
# -----------------------------
HOUSE_MAX_KW = 1.0
DATASET_MAX_KW = max(data['load'])  # update after checking dataset

# -----------------------------
# INPUT
# -----------------------------
input_data = {
    "hour": 9,
    "day": 2,
    "month": 4,
    "temperature": 35,
    "humidity": 50,
    "historical_load": 0.8
}

df = pd.DataFrame([input_data])

# -----------------------------
# PREDICT
# -----------------------------
raw_load = model.predict(df)[0]

scaled_load = (raw_load / DATASET_MAX_KW) * HOUSE_MAX_KW
scaled_load = max(0, min(scaled_load, HOUSE_MAX_KW))

# -----------------------------
# OUTPUT
# -----------------------------
print(f"\n🔹 Raw Load: {round(raw_load,2)} kW")
print(f"⚡ Scaled Load (House): {round(scaled_load,2)} kW")


🔹 Raw Load: 4.85 kW
⚡ Scaled Load (House): 0.58 kW


In [27]:
import joblib
import pandas as pd

model = joblib.load("models/load.pkl")

scenarios = [
    ("🌅 Morning Peak", 7),
    ("🌞 Afternoon", 14),
    ("🌆 Evening Peak", 20),
    ("🌙 Night", 2)
]

for name, hour in scenarios:

    df = pd.DataFrame([{
        "hour": hour,
        "day": 1,
        "month": 5,
        "temperature": 32,
        "humidity": 70,
        "historical_load": 2.5
    }])

    load = model.predict(df)[0]

    print(f"\n{name}")
    print(f"Load: {round(load,2)} kW")


🌅 Morning Peak
Load: 5.22 kW

🌞 Afternoon
Load: 3.96 kW

🌆 Evening Peak
Load: 6.5 kW

🌙 Night
Load: 4.14 kW
